# PHASE 1 — MACHINE LEARNING FOUNDATIONS


# Day 04 — Preprocessing


## 1. Learning Objectives
By the end of this notebook, you will be able to:
- Handle missing values using `SimpleImputer`.
- Encode categorical variables using `OneHotEncoder` and `OrdinalEncoder`.
- Scale numerical variables using `StandardScaler` and `MinMaxScaler`.
- Understand *why* scaling and encoding are mandatory for most machine learning models.


## 2. Prerequisites
- Basic Pandas DataFrame manipulation.
- Train/Test Splitting concepts (from Day 3).


## 3. Concept
Machine Learning models are essentially giant mathematical equations. You cannot multiply a string like `'Red'` by a weight of `3.5`. Furthermore, if one feature is in the thousands (e.g., Salary) and another is a fraction (e.g., Interest Rate), the larger number will dominate the equation, even if it is less important.

**Preprocessing** is the act of converting raw, messy, human-readable data into clean, scaled, numerical tensors that mathematical algorithms can process.


## 4. Why Does This Matter?
If you feed unscaled or unencoded data to Scikit-learn, most models (like Logistic Regression or Neural Networks) will either:
1. Throw an error (because they can't handle strings or NaNs).
2. Silently perform horribly (because unscaled features destroy the optimization process).


## 5. Intuition
- **Imputation**: Filling in the blanks. If a student missed a test, do we give them a 0, or do we give them their average score?
- **Encoding**: Translating languages. The model speaks Math. We must translate the word `'Paris'` into the array `[1, 0, 0]`.
- **Scaling**: Changing the measuring stick. Comparing a 5km run to a 5000m run is confusing. We must convert them to the same unit so they are easily comparable.


## 6. Mathematical Foundation
Consider Distance-based algorithms like K-Nearest Neighbors (which uses Euclidean distance):

$$ d(p, q) = \sqrt{\sum_{i=1}^{n} (p_i - q_i)^2} $$

If Feature 1 ranges from 0 to 1, and Feature 2 ranges from 0 to 1,000,000, the $(p_2 - q_2)^2$ term will be immensely huge. The distance calculation will completely ignore Feature 1. 

**StandardScaler** transforms features such that they have $\mu = 0$ and $\sigma = 1$:

$$ x_{scaled} = \frac{x - \mu}{\sigma} $$


## 7. Scikit-learn API
Preprocessing tools in Scikit-learn are called **Transformers**. They follow the exact same API:
1. Initialize: `scaler = StandardScaler()`
2. Learn parameters from training data: `scaler.fit(X_train)`
3. Apply to training data: `X_train_scaled = scaler.transform(X_train)`
4. Apply to test data: `X_test_scaled = scaler.transform(X_test)`


## 8. Simple Example
Let's process a messy dataset containing missing values, categorical strings, and unscaled numbers.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Create messy DataFrame
df = pd.DataFrame({
    'Age': [25, np.nan, 30, 45, 50],
    'Salary': [50000, 60000, 55000, 100000, np.nan],
    'City': ['Paris', 'London', 'London', 'New York', 'Paris'],
    'Target': [0, 1, 0, 1, 1]
})

X = df.drop('Target', axis=1)
y = df['Target']

# 1. Split FIRST to prevent leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

# 2. Impute (Fill missing values)
imputer = SimpleImputer(strategy='mean')
X_train_num = imputer.fit_transform(X_train[['Age', 'Salary']])
X_test_num = imputer.transform(X_test[['Age', 'Salary']])

print('Imputed Training Numbers:\n', X_train_num)


## 9. Code Walkthrough
- We created a dataframe with `np.nan` (missing values) and string categories.
- **Crucial step**: We split the data BEFORE doing any imputation. Why? If we calculated the mean of `Age` using the entire dataset, information from the test set would 'leak' into our training process!
- `SimpleImputer(strategy='mean')`: Learns the mean of the training data and fills the `NaN`s.


## 10. Experiment
Now let's apply `OneHotEncoder` to the 'City' column and `StandardScaler` to the numbers.


In [ ]:
# 3. Scale numerical data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_num)
X_test_scaled = scaler.transform(X_test_num)

# 4. Encode categorical data
encoder = OneHotEncoder(sparse_output=False)
X_train_cat = encoder.fit_transform(X_train[['City']])
X_test_cat = encoder.transform(X_test[['City']])

print('Scaled Numbers:\n', X_train_scaled)
print('\nOne-Hot Encoded Categories:\n', X_train_cat)
print('\nCategories learned by encoder:', encoder.categories_)


> `OneHotEncoder` converted 'London', 'New York', and 'Paris' into binary arrays (0s and 1s).


## 11. Prediction Exercise
Read the following code, but **DO NOT RUN IT YET**.


In [ ]:
from sklearn.preprocessing import OrdinalEncoder
size_data = pd.DataFrame({'Size': ['Small', 'Large', 'Medium', 'Small']})
ord_encoder = OrdinalEncoder(categories=[['Small', 'Medium', 'Large']])
encoded_sizes = ord_encoder.fit_transform(size_data)


> **Question:** What array will `encoded_sizes` produce?

**Think before running the next cell!**


In [ ]:
print(encoded_sizes)
print('\nWhy? OrdinalEncoder maps categories to ordered integers (0, 1, 2) based on the order provided in `categories`. OneHotEncoder would have created 3 separate columns.')


## 12. Coding Exercise
Use `MinMaxScaler` on the following data. `MinMaxScaler` scales data so it falls exactly between 0 and 1: $X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}}$.


In [ ]:
from sklearn.preprocessing import MinMaxScaler
data_to_scale = np.array([[10], [20], [30], [40], [50]])

# YOUR CODE HERE
minmax = MinMaxScaler()
scaled_data = minmax.fit_transform(data_to_scale)
print('MinMax Scaled:\n', scaled_data)


## 13. Debugging Challenge
The junior data scientist tried to scale the data, but caused massive Data Leakage. Find the bug!


In [ ]:
# Buggy code
try:
    bad_scaler = StandardScaler()
    X_bad = df[['Age', 'Salary']].dropna() # Ignoring NaNs for a moment
    y_bad = df.loc[X_bad.index, 'Target']
    
    # THE BUG IS HERE:
    X_bad_scaled = bad_scaler.fit_transform(X_bad)
    X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(X_bad_scaled, y_bad)
    
    print('Data scaled successfully!')
except Exception as e:
    print('Error:', e)


> **Hint:** Look at the order of operations. Did the scaler see the test data?


## 14. Model Evaluation
Why do we have two encoders?
- **OneHotEncoder**: Use for *Nominal* data (no inherent order). E.g., Red, Green, Blue. London, Paris.
- **OrdinalEncoder**: Use for *Ordinal* data (inherent order). E.g., Low, Medium, High. Bad, Good, Excellent.
If you use Ordinal encoding on cities (London=0, Paris=1, NY=2), the model will mathematically assume NY is 'greater than' London, which makes no sense!


## 15. Real-World Example
In real-world APIs, unseen categories crash models. If you deploy a model in Paris and London, and tomorrow someone enters 'Tokyo', `OneHotEncoder` will throw an error.
To fix this, you must initialize it with `OneHotEncoder(handle_unknown='ignore')` so it outputs all zeros for unseen categories instead of crashing the server.


## 16. Mini Project
Combine the preprocessed arrays from Section 10 back together into a single training set using `np.hstack()`.


In [ ]:
# Combine scaled numbers and encoded categories
X_train_final = np.hstack([X_train_scaled, X_train_cat])
X_test_final = np.hstack([X_test_scaled, X_test_cat])

print('Final Training Array Shape:', X_train_final.shape)
print('Final Test Array Shape:', X_test_final.shape)
print('\nFirst row of fully processed training data:\n', X_train_final[0])


## 17. Common Mistakes
- **Data Leakage**: Fitting a scaler/imputer on the entire dataset before splitting.
- **Transforming train data twice**: Doing `fit_transform` on train, then mistakenly doing `fit_transform` on test.
- **Not saving the scaler**: If you train a model, you MUST save the fitted scaler. Without it, you cannot scale real-world inference data identically!


## 18. Interview Questions
- **Beginner**: What is the difference between `StandardScaler` and `MinMaxScaler`?
- **Intermediate**: Why do we use `OneHotEncoder` instead of `OrdinalEncoder` for country names?
- **Advanced**: Explain how computing the mean for imputation before `train_test_split` causes data leakage.


## 19. Knowledge Check
- What function combines fitting and transforming into one step? (`fit_transform`)
- How do you handle missing numeric data? (`SimpleImputer`)


## 20. Summary
- Preprocessing is mandatory for ML.
- **Impute** missing values first.
- **Encode** categorical variables into numbers.
- **Scale** numerical variables so they are mathematically comparable.
- ALWAYS split data *before* fitting any preprocessors to avoid data leakage.


## 21. Homework
Create a DataFrame with missing values. Split it. Use `SimpleImputer(strategy='median')` on the training set, and then `.transform()` the test set. Verify the NaNs are gone.
